# Домашня робота на тему "Поліноміальна регресія та регуляризація"

Цей ноутбук містить завдання Домашньої роботи з використанням датасету California Housing. Розглядаються поліноміальна регресія різних ступенів, регуляризація Lasso та Ridge.

## Імпорт бібліотек та завантаження даних

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import r2_score

# Завантаження датасету
housing = fetch_california_housing()
X = housing.data
y = housing.target
feature_names = housing.feature_names

# Створення DataFrame для зручності
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print("Розмір датасету:", df.shape)
df.head()

Розмір датасету: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## Завдання 1: Поліноміальна регресія ступеня 2 на ознаці MedInc

**Умова:**
1. Використовувати тільки одну ознаку MedInc.
2. Створити поліноміальні ознаки ступеня 2.
3. Розділити дані на train/test (80/20, random_state=42).
4. Навчити модель і обчислити R² на тестовій вибірці.

In [7]:
# Виділення ознаки MedInc
X_medinc = df[['MedInc']].values
y = df['target'].values

# Розділення
X_train, X_test, y_train, y_test = train_test_split(
    X_medinc, y, test_size=0.2, random_state=42)
# Поліноміальні ознаки ступеня 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train = poly.fit_transform(X_train)
X_test = poly.transform(X_test)


# Модель
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"R² на тестовій вибірці для поліноміальної регресії ступеня 2: {r2:.4f}")

R² на тестовій вибірці для поліноміальної регресії ступеня 2: 0.4633


## Завдання 2: Вплив ступеня полінома на якість моделі

**Умова:**
1. Використовувати ознаку MedInc.
2. Побудувати моделі з PolynomialFeatures ступенів 1, 2 і 3.
3. Для кожної моделі обчислити R² на train і test.
4. Вивести результати в таблиці.

In [13]:
X_medinc = df[['MedInc']].values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_medinc, y, test_size=0.2, random_state=42
)

poly1 = PolynomialFeatures(degree=1, include_bias=False)
X_train1 = poly1.fit_transform(X_train)
X_test1 = poly1.transform(X_test)

model1 = LinearRegression()
model1.fit(X_train1, y_train)
y_pred = model1.predict(X_test1)
r2_1 = r2_score(y_test, y_pred)
r2_1x = r2_score(y_train, model1.predict(X_train1))

poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_train2 = poly2.fit_transform(X_train)
X_test2 = poly2.transform(X_test)

model2 = LinearRegression()
model2.fit(X_train2, y_train)
y_pred = model2.predict(X_test2)
r2_2 = r2_score(y_test, y_pred)
r2_2x = r2_score(y_train, model2.predict(X_train2))

poly3 = PolynomialFeatures(degree=3, include_bias=False)
X_train3 = poly3.fit_transform(X_train)
X_test3 = poly3.transform(X_test)

model3 = LinearRegression()
model3.fit(X_train3, y_train)
y_pred = model3.predict(X_test3)
r2_3 = r2_score(y_test, y_pred)
r2_3x = r2_score(y_train, model3.predict(X_train3))

results = pd.DataFrame({
    "Model": ["Degree 1", "Degree 2", "Degree 3"],
    "R^2 Test": [r2_1, r2_2, r2_3],
    "R^2 Train": [r2_1x, r2_2x, r2_3x],
})
display(results)

,Model,R^2 Test,R^2 Train
0,Degree 1,0.458859,0.476993
1,Degree 2,0.463318,0.481598
2,Degree 3,0.467116,0.490806


## Завдання 3: Lasso-регуляризація vs лінійна регресія

**Умова:**
1. Використовувати датасет California Housing.
2. Взяти ознаки MedInc, HouseAge, AveRooms, AveBedrms.
3. Побудувати дві моделі: LinearRegression та Lasso(alpha=0.1).
4. Розділити дані на train/test (80/20, random_state=42).
5. Для кожної моделі обчислити R² на тестовій вибірці.
6. Порівняти коефіцієнти моделей: скільки з них стали рівними 0 у Lasso.

In [23]:
# Вибір ознак
features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms']
X_selected = df[features].values
y = df['target'].values

# Розділення
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42)

# Лінійна регресія
model_lin = LinearRegression()
model_lin.fit(X_train, y_train)
y_pred = model_lin.predict(X_test)
r2_lin = r2_score(y_test, y_pred)

# Lasso
model_las = Lasso(alpha=0.1)
model_las.fit(X_train, y_train)
y_pred = model_las.predict(X_test)
r2_las = r2_score(y_test, y_pred)

# Кількість нульових коефіцієнтів у Lasso
lin_coef = model_lin.coef_
las_coef = model_las.coef_


# Порівняння коефіцієнтів
zero_count =0
for i in range(0,len(las_coef)):
    if las_coef[i] == 0:
        zero_count += 1
        # Alternative way. Somehow i think python first and then look for other options :)
        # By the way, this what i found most intriguing in data science. Even though we r using the same language
        # But it just u can write less code and get much more results. Like pipeline is so powerful on its own. Love this stuff!!!
        # Thank you!!!
zero_count_alt = np.sum(las_coef == 0)

results = pd.DataFrame({
    "Features": features,
    "Linear": [lin_coef[0], lin_coef[1], lin_coef[2], lin_coef[3]],
    "Lasso": [las_coef[0], las_coef[1], las_coef[2], las_coef[3]],
})
display(results.round(4))
print(f"Zero count: {zero_count}")

,Features,Linear,Lasso
0,MedInc,0.5453,0.4051
1,HouseAge,0.0161,0.0163
2,AveRooms,-0.2246,-0.0004
3,AveBedrms,1.1130,0.0000


Zero count: 1


## Завдання 4: Ridge-регуляризація vs лінійна регресія

**Умова:**
1. Використовувати ті самі ознаки (MedInc, HouseAge, AveRooms, AveBedrms).
2. Побудувати дві моделі: LinearRegression та Ridge(alpha=1.0).
3. Розділити дані на train/test (80/20, random_state=42).
4. Для кожної моделі обчислити R² на тестовій вибірці.
5. Оформити результати в таблиці з кількістю ненульових коефіцієнтів.

In [29]:
# Повторне розділення (для узгодженості)
features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms']

X_selected = df[features].values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42)

# Ridge
model_ridge = Ridge(alpha=1.0)
model_ridge.fit(X_train, y_train)
y_pred = model_ridge.predict(X_test)
r2_ridge = r2_score(y_test, y_pred)

# Повторна лінійна регресія для таблиці
model_lin1 = LinearRegression()
model_lin1.fit(X_train, y_train)
y_pred = model_lin1.predict(X_test)
r2_lin = r2_score(y_test, y_pred)

# Кількість ненульових коефіцієнтів
ridge_coef = model_ridge.coef_
lin1_coef = model_ridge.coef_
none_zero_ridge = np.sum(ridge_coef != 0)
none_zero_lin = np.sum(lin1_coef != 0)
none_zero_las = np.sum(las_coef != 0)
zero_ridge = np.sum(ridge_coef == 0)
zero_lin = np.sum(lin1_coef == 0)


# Таблиця
results = pd.DataFrame({
    "Model": ["Linear", "Lasso", "Ridge"],
    "R^2 Test": [r2_lin, r2_las, r2_ridge],
    "None Zero Coefficients": [none_zero_lin, none_zero_las, none_zero_ridge],
    "Zero Coefficients": [zero_lin, zero_count_alt, zero_ridge],
})
display(results.round(4))

,Model,R^2 Test,None Zero Coefficients,Zero Coefficients
0,Linear,0.5089,4,0
1,Lasso,0.4933,3,1
2,Ridge,0.5090,4,0
